In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer, DataCollatorForLanguageModeling
from peft import LoraConfig, get_peft_model
from torch.utils.data import Dataset

# ==========================================
# 1. CARREGAR O CÉREBRO E O TRADUTOR
# ==========================================
nome_do_modelo = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"
tokenizer = AutoTokenizer.from_pretrained(nome_do_modelo)

# O TinyLlama não tem um "token de preenchimento" padrão para treinos curtos,
# então ensinamos ele a usar o token de "fim de frase" para isso.
tokenizer.pad_token = tokenizer.eos_token

print("Carregando o modelo base para a memória...")
modelo_base = AutoModelForCausalLM.from_pretrained(nome_do_modelo)

# ==========================================
# 2. CONFIGURAR O PEN-DRIVE LORA
# ==========================================
configuracao_lora = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    bias="none",
    task_type="CAUSAL_LM"
)
modelo_para_treinar = get_peft_model(modelo_base, configuracao_lora)

# ==========================================
# 3. PREPARAR OS DADOS (O seu conhecimento.txt)
# ==========================================
class MeuDatasetPersonalizado(Dataset):
    def __init__(self, caminho_arquivo, tokenizer, tamanho_maximo=128):
        with open(caminho_arquivo, 'r', encoding='utf-8') as f:
            # Lê todas as linhas do seu arquivo, ignorando linhas em branco
            self.linhas = [linha.strip() for linha in f if linha.strip()]
        self.tokenizer = tokenizer
        self.tamanho_maximo = tamanho_maximo

    def __len__(self):
        return len(self.linhas)

    def __getitem__(self, idx):
        texto = self.linhas[idx]

        # Converte a sua frase em Português para os IDs matemáticos da IA
        codificacao = self.tokenizer(
            texto,
            truncation=True,
            padding="max_length",
            max_length=self.tamanho_maximo,
            return_tensors="pt"
        )

        # Como a IA tenta prever a própria frase, os 'labels' (respostas corretas)
        # são iguais aos 'input_ids' (a frase lida).
        item = {chave: valor.squeeze(0) for chave, valor in codificacao.items()}
        item["labels"] = item["input_ids"].clone()
        return item

print("Formatando o dataset de conhecimento...")
meu_dataset = MeuDatasetPersonalizado("conhecimento.txt", tokenizer)
colator_de_dados = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# ==========================================
# 4. CONFIGURAR O MOTOR DE TREINO (Hugging Face Trainer)
# ==========================================
config_treino = TrainingArguments(
    output_dir="./resultados_lora", # Pasta onde ele salva backups temporários
    num_train_epochs=50,            # Quantas vezes ele vai ler o seu texto
    per_device_train_batch_size=2,  # Lê 2 frases por vez
    save_steps=50,                  # Faz um backup a cada 50 passos
    logging_steps=5,                # Mostra o Loss a cada 5 passos no terminal
    learning_rate=2e-4,             # Velocidade de aprendizado padrão para LoRA
    optim="adamw_torch"
)

treinador = Trainer(
    model=modelo_para_treinar,
    args=config_treino,
    train_dataset=meu_dataset,
    data_collator=colator_de_dados
)

# ==========================================
# 5. INICIAR A SALA DE AULA
# ==========================================
print("\n🚀 Iniciando o Treinamento Neural...")
treinador.train()

# 6. SALVAR O PEN-DRIVE FINAL
# Ele não salva o modelo de 2GB de novo, ele salva SÓ a camada do LoRA (uns 10 MB!)
modelo_para_treinar.save_pretrained("./meu_chatbot_tinyllama")
print("\n✅ Treinamento concluído! O seu modelo foi salvo na pasta './meu_chatbot_tinyllama'.")

Carregando o modelo base para a memória...


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Formatando o dataset de conhecimento...

🚀 Iniciando o Treinamento Neural...
{'loss': '6.144', 'grad_norm': '4.884', 'learning_rate': '0.000198', 'epoch': '0.625'}
{'loss': '5.728', 'grad_norm': '4.62', 'learning_rate': '0.0001955', 'epoch': '1.25'}
{'loss': '5.386', 'grad_norm': '3.198', 'learning_rate': '0.000193', 'epoch': '1.875'}
{'loss': '4.956', 'grad_norm': '4.105', 'learning_rate': '0.0001905', 'epoch': '2.5'}
{'loss': '4.497', 'grad_norm': '4.076', 'learning_rate': '0.000188', 'epoch': '3.125'}
{'loss': '4.207', 'grad_norm': '3.602', 'learning_rate': '0.0001855', 'epoch': '3.75'}
{'loss': '3.822', 'grad_norm': '3.859', 'learning_rate': '0.000183', 'epoch': '4.375'}
{'loss': '3.673', 'grad_norm': '5.444', 'learning_rate': '0.0001805', 'epoch': '5'}
{'loss': '3.474', 'grad_norm': '3.804', 'learning_rate': '0.000178', 'epoch': '5.625'}
{'loss': '2.968', 'grad_norm': '3.877', 'learning_rate': '0.0001755', 'epoch': '6.25'}
{'loss': '2.701', 'grad_norm': '5.818', 'learning_rate': '

In [ ]:
# Instale a biblioteca de datasets se o Colab pedir
# !pip install datasets

from datasets import load_dataset

# 1. BAIXANDO O DATASET DA NUVEM
print("📥 Baixando 5.000 conversas em português do Hugging Face...")
# O split="train[:5000]" garante que não vamos pegar as 50k linhas de uma vez
dataset = load_dataset("dominguesm/alpaca-data-pt-br", split="train[:5000]")

nome_arquivo = "conhecimento_gigante.txt"

print("⚙️ Formatando para o cérebro da nossa IA...")
with open(nome_arquivo, "w", encoding="utf-8") as f:
    for item in dataset:
        pergunta = item['instruction'].strip()
        # Algumas perguntas têm um contexto extra no campo 'input'
        if item['input']:
            pergunta += " " + item['input'].strip()

        resposta = item['output'].strip()

        # Remove quebras de linha para não bugar o nosso código
        pergunta = pergunta.replace('\n', ' ')
        resposta = resposta.replace('\n', ' ')

        # Escreve no formato exato que a nossa IA espera
        f.write(f"<pergunta> {pergunta.lower()} <resposta> {resposta.lower()}\n")

print(f"✅ Sucesso! Arquivo '{nome_arquivo}' criado com 5.000 exemplos de treinamento.")

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import os

# CONFIGURAÇÕES TÉCNICAS
ARQUIVO_TREINO = 'conhecimento_gigante.txt'
ARQUIVO_PESOS = 'chatbot_pesos.pth'
DISPOSITIVO = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 1. CARREGAMENTO E VOCABULÁRIO ROBUSTO
if not os.path.exists(ARQUIVO_TREINO):
    with open(ARQUIVO_TREINO, 'w', encoding='utf-8') as f:
        f.write("<pergunta> ola <resposta> oi, sou a ia do ellias\n")

with open(ARQUIVO_TREINO, 'r', encoding='utf-8') as f:
    text = f.read().lower()

words = text.split()
palavras_unicas = sorted(list(set(words)))

# O SEGREDO: Reservamos o ID 0 para palavras que a IA não conhece (<UNK>)
vocab = {palavra: i + 1 for i, palavra in enumerate(palavras_unicas)}
vocab['<UNK>'] = 0
itos = {i: palavra for palavra, i in vocab.items()}

# 2. ARQUITETURA DO MODELO
class ChatbotPro(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.rnn = nn.RNN(embedding_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x, hidden):
        embedded = self.embedding(x)
        output, hidden = self.rnn(embedded, hidden)
        return self.fc(output), hidden

    def init_hidden(self, batch_size):
        return torch.zeros(1, batch_size, self.hidden_dim).to(DISPOSITIVO)

# Instanciando
model = ChatbotPro(len(vocab), 16, 32).to(DISPOSITIVO)

# 3. LÓGICA DE PERSISTÊNCIA (SALVAR/CARREGAR)
def treinar():
    print(f"Ensinando {len(words)} palavras para a IA...")
    optimizer = optim.Adam(model.parameters(), lr=0.005)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(300):
        total_loss = 0
        hidden = model.init_hidden(1)
        for i in range(len(words) - 3):
            # Janela deslizante de 3 palavras
            contexto = [vocab[w] for w in words[i:i+3]]
            alvo = vocab[words[i+3]]

            inputs = torch.tensor([contexto]).to(DISPOSITIVO)
            target = torch.tensor([alvo]).to(DISPOSITIVO)

            optimizer.zero_grad()
            output, _ = model(inputs, hidden)
            loss = criterion(output[0, -1, :].unsqueeze(0), target)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        if (epoch + 1) % 50 == 0:
            print(f"Época {epoch+1} | Erro: {total_loss:.4f}")

    torch.save(model.state_dict(), ARQUIVO_PESOS)
    print("Cérebro salvo com sucesso!")

if os.path.exists(ARQUIVO_PESOS):
    print("Carregando memória do HD...")
    model.load_state_dict(torch.load(ARQUIVO_PESOS))
else:
    treinar()

# 4. CHAT COM CONTEXTO E UNK
def responder(frase):
    model.eval()
    tokens = [vocab.get(w, 0) for w in f"<pergunta> {frase.lower()} <resposta>".split()]

    # SALVAMOS O TAMANHO DA PERGUNTA ORIGINAL
    tamanho_pergunta = len(tokens)

    for _ in range(15): # Limite de palavras na resposta
        input_tensor = torch.tensor([tokens]).to(DISPOSITIVO)
        output, _ = model(input_tensor, model.init_hidden(1))
        idx = torch.argmax(output[0, -1, :]).item()

        if itos.get(idx) == '<pergunta>' or itos.get(idx) == '\n': break
        tokens.append(idx)

    # O SEGREDO: Pegamos apenas os tokens gerados DEPOIS da pergunta!
    tokens_resposta = tokens[tamanho_pergunta:]
    return ' '.join([itos[i] for i in tokens_resposta if itos[i] not in ['<pergunta>', '<resposta>']])

import os

print("\n=======================================")
print("🧠 IA PRONTA E ATIVA")
print("💡 Digite '/ensinar' para adicionar novos conhecimentos.")
print("🚪 Digite 'sair' para encerrar.")
print("=======================================")

while True:
    user = input("\nVocê: ")

    if user.lower() == 'sair':
        break

    # O SISTEMA DE AUTOALIMENTAÇÃO
    elif user.lower() == '/ensinar':
        print("\n--- MODO DE APRENDIZADO ---")
        nova_pergunta = input("Quando o usuário perguntar: ")
        nova_resposta = input("A IA deve responder: ")

        # 1. Salva no arquivo de texto
        with open(ARQUIVO_TREINO, 'a', encoding='utf-8') as f:
            f.write(f"\n<pergunta> {nova_pergunta.lower()} <resposta> {nova_resposta.lower()}")

        # 2. Destrói o cérebro antigo para forçar o retreino!
        if os.path.exists(ARQUIVO_PESOS):
            os.remove(ARQUIVO_PESOS)

        print("✅ Anotado no banco de dados!")
        print("⚠️ Feche o programa e rode novamente para eu treinar meu novo cérebro.")
        continue

    # FLUXO NORMAL DE CONVERSA
    else:
        print(f"IA: {responder(user)}")

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import warnings
from transformers import logging

warnings.filterwarnings("ignore")
logging.set_verbosity_error() # Força a biblioteca a mostrar apenas erros fatais, escondendo os avisos

# ==========================================
# 1. CARREGAR AS PEÇAS
# ==========================================
nome_modelo_base = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"
caminho_pen_drive = "./meu_chatbot_tinyllama"

print("🧠 Carregando o Cérebro Base (TinyLlama)...")
tokenizer = AutoTokenizer.from_pretrained(nome_modelo_base)
modelo_base = AutoModelForCausalLM.from_pretrained(nome_modelo_base)

print("🔌 Plugando o seu Conhecimento (LoRA)...")
# A MÁGICA ACONTECE AQUI: Fundimos o modelo base com o seu treinamento!
modelo_ia = PeftModel.from_pretrained(modelo_base, caminho_pen_drive)

# Opcional: Se quiser que ela responda mais rápido na CPU, usamos torch.bfloat16
modelo_ia = modelo_ia.to(torch.float32)

# ==========================================
# 2. O LOOP DE CONVERSA
# ==========================================
def conversar(pergunta_usuario):
    regras = (
        "Você é o assistente técnico do Ellias.\n"
        "REGRAS OBRIGATÓRIAS:\n"
        "- O banco é Supabase.\n"
        "- Chaves primárias DEVEM começar com 'cod_' (ex: cod_maquina).\n"
        "- Use apenas uma tabela para todas as máquinas.\n"
    )

    prompt = f"{regras}\n<pergunta> {pergunta_usuario.lower()} <think>"
    inputs = tokenizer(prompt, return_tensors="pt")

    with torch.no_grad():
        outputs = modelo_ia.generate(
            **inputs,
            max_new_tokens=50,

            # --- AS CORREÇÕES ESTÃO AQUI ---
            do_sample=True,          # Permite usar temperatura
            temperature=0.3,         # 0.7 dá um pouco de "criatividade" sem pirar
            repetition_penalty=1.2,  # Punição matemática para ela não se repetir
            top_p=1,               # Filtra as palavras mais prováveis
            pad_token_id=tokenizer.eos_token_id
        )

    resposta_completa = tokenizer.decode(outputs[0], skip_special_tokens=True)

    if "<resposta>" in resposta_completa:
        texto_ia = resposta_completa.split("<resposta>")[-1].strip()
    elif "</think>" in resposta_completa:
        texto_ia = resposta_completa.split("</think>")[-1].strip()
    else:
        texto_ia = resposta_completa.split("<pergunta>")[-1].split("?")[-1].strip()

    # 4. LIMPEZA FINAL
    for stop in ["\n", "Você:", "IA:", "<pergunta>"]:
        texto_ia = texto_ia.split(stop)[0]

    return texto_ia.strip()

print("\n=======================================")
print("🤖 IA LOCAL INICIADA (CPU Mode)")
print("=======================================")

while True:
    texto = input("\nVocê: ")
    if texto.lower() == 'sair':
        break

    print(f"IA: {conversar(texto.lower())}")

🧠 Carregando o Cérebro Base (TinyLlama)...


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

🔌 Plugando o seu Conhecimento (LoRA)...

🤖 IA LOCAL INICIADA (CPU Mode)
IA: ola <think>
IA: como devo nomear a chave primaria <think>


In [6]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import warnings
from transformers import logging

warnings.filterwarnings("ignore")
logging.set_verbosity_error()

# ==========================================
# 1. CARREGAR O CÉREBRO (Modelo Aberto, sem senha)
# ==========================================
nome_do_modelo = "Qwen/Qwen2.5-1.5B-Instruct"
# nome_do_modelo =  "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"

print("🚀 Carregando a IA e preparando o Motor de Raciocínio...")
tokenizer = AutoTokenizer.from_pretrained(nome_do_modelo)

# Usamos bfloat16 ou float16 para rodar rápido e não estourar a RAM
modelo_ia = AutoModelForCausalLM.from_pretrained(
    nome_do_modelo,
    torch_dtype=torch.float16,
    device_map="auto" # Usa a GPU automaticamente se estiver no Colab
)

# ==========================================
# 2. A ENGRENAGEM DE RACIOCÍNIO
# ==========================================
def pensar_e_responder(pergunta_usuario):
    prompt_sistema = (
        "Você é o assistente pessoal do Ellias. "
        "REGRA ABSOLUTA: Você DEVE estruturar a sua saída EXATAMENTE neste formato:\n"
        "<think> seu raciocínio passo a passo aqui </think>\n"
        "A sua resposta final e direta para o usuário aqui."
    )

    # 1. A ESTRUTURA PROFISSIONAL (O Uniforme)
    # Em vez de colar texto puro, nós definimos os "papéis" da conversa
    mensagens = [
        {"role": "system", "content": prompt_sistema},
        {"role": "user", "content": pergunta_usuario}
    ]

    # 2. O TRADUTOR DE TAGS SECRETAS
    # Isso transforma o dicionário acima no formato exato que o Qwen foi treinado para ler
    prompt_formatado = tokenizer.apply_chat_template(
        mensagens,
        tokenize=False,
        add_generation_prompt=True # Prepara a IA para ser a próxima a falar
    )

    # 3. A NOSSA ARMADILHA
    # Colamos o nosso hack de raciocínio logo após a IA ser "convocada" a falar
    prompt_completo = prompt_formatado + "<think>\n"

    inputs = tokenizer(prompt_completo, return_tensors="pt").to(modelo_ia.device)

    with torch.no_grad():
        outputs = modelo_ia.generate(
            **inputs,
            max_new_tokens=400,
            do_sample=True,
            temperature=0.4,
            repetition_penalty=1.0,
            pad_token_id=tokenizer.eos_token_id
        )

    texto_bruto = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # ... (O resto do código de extração continua EXATAMENTE igual) ...
    pensamento = "A IA respondeu sem pensar."
    resposta_final = texto_bruto

    texto_gerado = texto_bruto.split("<think>")[-1]

    if "</think>" in texto_gerado:
        partes = texto_gerado.split("</think>")
        pensamento = partes[0].strip()
        resposta_final = partes[1].replace("<resposta>", "").replace("</resposta>", "").strip()

    return pensamento, resposta_final

# ==========================================
# 3. O LOOP DO USUÁRIO
# ==========================================
print("\n==================================================")
print("🧠 IA COM RACIOCÍNIO LÓGICO INICIADA")
print("==================================================")

while True:
    texto = input("\nVocê: ")
    if texto.lower() == 'sair':
        break

    # Chama a função que faz o trabalho pesado
    pensamento_ia, resposta_ia = pensar_e_responder(texto)

    # Imprime o pensamento em CINZA ESCURO (código ANSI) para ficar com cara de "bastidores"
    print("\n\033[90m[LÓGICA INTERNA DA IA]:")
    print(f"{pensamento_ia}\033[0m")

    # Imprime a resposta final normal
    print(f"\n🤖 Resposta Final:\n{resposta_ia}")
    print("-" * 50)

🚀 Carregando a IA e preparando o Motor de Raciocínio...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]


🧠 IA COM RACIOCÍNIO LÓGICO INICIADA

Você: ola, como você está?

[LÓGICA INTERNA DA IA]:
Olá! Estou bem, obrigado por perguntar. Como posso ajudar você hoje?

🤖 Resposta Final:
A sua resposta final e direta para o usuário aqui: Estou bem, obrigado por perguntar. Como posso ajudar você hoje?
--------------------------------------------------

Você: quero saber quanto é 10 + 10

[LÓGICA INTERNA DA IA]:
Para resolver a soma 10 + 10, basta somar os dois números. O resultado é 20.

🤖 Resposta Final:
A resposta é 20.
--------------------------------------------------

Você: pode me explicar como napoleão ficou tão famoso

[LÓGICA INTERNA DA IA]:
Napoleão Bonaparte se tornou famoso por várias razões. Primeiro, ele foi um general excepcional, liderando a França durante o período de guerra napoleônica. Seu sucesso militar foi impressionante, conquistando territórios e estabelecendo um império em grande parte da Europa. 

Em segundo lugar, Napoleão teve uma personalidade forte e influente. Ele 

KeyboardInterrupt: Interrupted by user


Você: 


In [8]:
# pip install duckduckgo-search

In [8]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import warnings
from transformers import logging

# 1. BLOQUEIO TOTAL DE AVISOS (Paz no terminal)
warnings.filterwarnings("ignore")
warnings.filterwarnings("ignore", category=DeprecationWarning)
logging.set_verbosity_error()

try:
    from ddgs import DDGS # Usando a biblioteca nova
except ImportError:
    from duckduckgo_search import DDGS # Fallback caso você ainda use a velha

# ==========================================
# 1. CARREGAR O CÉREBRO
# ==========================================
nome_do_modelo = "Qwen/Qwen2.5-1.5B-Instruct"

print("🚀 Carregando a IA e preparando o Motor de Raciocínio...")
tokenizer = AutoTokenizer.from_pretrained(nome_do_modelo)
modelo_ia = AutoModelForCausalLM.from_pretrained(
    nome_do_modelo,
    torch_dtype=torch.float16,
    device_map="auto"
)

# ==========================================
# 2. A FERRAMENTA DE BUSCA (BACKEND)
# ==========================================
def pesquisar_na_internet(termo):
    print(f"\n   [🌐 O Python pausou a IA e está pesquisando: '{termo}'...]")
    try:
        resultados = DDGS().text(termo, max_results=3)
        textos = [f"- Site: {r['title']}\n  Conteúdo: {r['body']}" for r in resultados]
        return "\n\n".join(textos)
    except Exception as e:
        return "Nenhum resultado encontrado ou erro na rede."

# ==========================================
# 3. A ENGRENAGEM DE RACIOCÍNIO BLINDADA
# ==========================================
def pensar_e_responder(pergunta_usuario, contexto_extra="", ja_pesquisou=False):

    # 1. O PROMPT MODO "CAVEMAN" (Curto, Direto e Agressivo)
    if not ja_pesquisou:
        prompt_sistema = (
            "Você é o assistente de busca.\n"
            "REGRA 1: Você NÃO SABE informações de eventos recentes, cotações ou notícias.\n"
            "REGRA 2: VOCÊ TEM SIM ACESSO À INTERNET. Tudo que requisitar pesquisas, pesquise.\n"
            "REGRA 3: Quando não souber, use EXATAMENTE a tag [PESQUISAR] e NADA MAIS na resposta final.\n\n"
            "EXEMPLOS OBRIGATÓRIOS DE COMO VOCÊ DEVE AGIR:\n"
            "Usuário: Qual a cotação do dólar hoje?\n"
            "Assistente: <think> Eu não sei a cotação de hoje, preciso acessar a internet. </think>\n[PESQUISAR] cotação dólar hoje\n\n"
            "Usuário: Notícias da SpaceX.\n"
            "Assistente: <think> Isso é um evento recente, não tenho na memória. </think>\n[PESQUISAR] SpaceX últimas notícias\n\n"
            "Usuário: Quanto é 2+2?\n"
            "Assistente: <think> Isso é matemática básica, eu sei a resposta. </think>\nA resposta é 4."
        )
    else:
        # PÓS-PESQUISA: Forçamos ela a sair do <think>
        prompt_sistema = (
            "Você é um assistente lógico.\n"
            "REGRA 1: Use a tag <think> APENAS para analisar os dados em silêncio.\n"
            "REGRA 2: Você JÁ RECEBEU os dados da internet. É PROIBIDO usar [PESQUISAR] novamente.\n"
            "REGRA 3: OBRIGATÓRIO: Você DEVE fechar a tag com </think> e escrever a resposta final ao usuário logo abaixo dela."
        )
        prompt_sistema += f"\n\n[TEXTO DA INTERNET]:\n{contexto_extra}"

    mensagens = [
        {"role": "system", "content": prompt_sistema},
        {"role": "user", "content": pergunta_usuario}
    ]

    prompt_formatado = tokenizer.apply_chat_template(mensagens, tokenize=False, add_generation_prompt=True)
    prompt_completo = prompt_formatado + "<think>\n"

    inputs = tokenizer(prompt_completo, return_tensors="pt").to(modelo_ia.device)

    with torch.no_grad():
        outputs = modelo_ia.generate(
            **inputs,
            max_new_tokens=400,
            do_sample=True,
            temperature=0.4,         # Reduzi para 0.3 para ela parar de alucinar "Cabo Verde"
            repetition_penalty=1.0,
            pad_token_id=tokenizer.eos_token_id
        )

    texto_bruto = tokenizer.decode(outputs[0], skip_special_tokens=True)
    pensamento = "Falha no raciocínio."
    resposta_final = texto_bruto
    texto_gerado = texto_bruto.split("<think>")[-1]

    if "</think>" in texto_gerado:
        partes = texto_gerado.split("</think>")
        pensamento = partes[0].strip()
        resposta_final = partes[1].replace("<resposta>", "").replace("</resposta>", "").strip()

        if resposta_final == "":
             resposta_final = "A IA respondeu apenas no pensamento: " + pensamento

    # ==========================================
    # 4. A TRAVA DE SEGURANÇA (MÁXIMO 1 PESQUISA)
    # ==========================================
    if "[PESQUISAR]" in resposta_final and not ja_pesquisou:
        termo_busca = resposta_final.split("[PESQUISAR]")[1].strip()
        termo_busca = termo_busca.replace(".", "").replace('"', '').replace(']', '')

        dados_da_web = pesquisar_na_internet(termo_busca)
        print("   [🧠 Dados capturados! Reinjetando na mente da IA e bloqueando novas buscas...]")

        # Chama a função de novo, mas passando ja_pesquisou=True (Trava o Loop)
        return pensar_e_responder(pergunta_usuario, contexto_extra=dados_da_web, ja_pesquisou=True)

    return pensamento, resposta_final

# ==========================================
# O LOOP
# ==========================================
print("\n==================================================")
print("🧠 IA COM RACIOCÍNIO E ACESSO À INTERNET INICIADA")
print("==================================================")

while True:
    texto = input("\nVocê: ")
    if texto.lower() == 'sair':
        break

    pensamento_ia, resposta_ia = pensar_e_responder(texto)

    print("\n\033[90m[LÓGICA INTERNA DA IA]:")
    print(f"{pensamento_ia}\033[0m")
    print(f"\n🤖 Resposta Final:\n{resposta_ia}")
    print("-" * 50)

🚀 Carregando a IA e preparando o Motor de Raciocínio...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]


🧠 IA COM RACIOCÍNIO E ACESSO À INTERNET INICIADA


KeyboardInterrupt: Interrupted by user

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import warnings
from transformers import logging

# 1. BLOQUEIO TOTAL DE AVISOS (Paz no terminal)
warnings.filterwarnings("ignore")
warnings.filterwarnings("ignore", category=DeprecationWarning)
logging.set_verbosity_error()

try:
    from ddgs import DDGS # Usando a biblioteca nova
except ImportError:
    from duckduckgo_search import DDGS # Fallback caso você ainda use a velha

# ==========================================
# 1. CARREGAR O CÉREBRO
# ==========================================
nome_do_modelo = "Qwen/Qwen2.5-1.5B-Instruct"

print("🚀 Carregando a IA e preparando o Motor de Raciocínio...")
tokenizer = AutoTokenizer.from_pretrained(nome_do_modelo)
modelo_ia = AutoModelForCausalLM.from_pretrained(
    nome_do_modelo,
    torch_dtype=torch.float16,
    device_map="auto"
)

# ==========================================
# 2. A FERRAMENTA DE BUSCA (BACKEND)
# ==========================================
def pesquisar_na_internet(termo):
    print(f"\n   [🌐 O Python pausou a IA e está pesquisando: '{termo}'...]")
    try:
        resultados = DDGS().text(termo, max_results=3)
        textos = [f"- Site: {r['title']}\n  Conteúdo: {r['body']}" for r in resultados]
        return "\n\n".join(textos)
    except Exception as e:
        return "Nenhum resultado encontrado ou erro na rede."

# ==========================================
# 3. A ENGRENAGEM DE RACIOCÍNIO BLINDADA
# ==========================================
# ==========================================
# 3. A ENGRENAGEM DE RACIOCÍNIO BLINDADA E COM MEMÓRIA
# ==========================================
def pensar_e_responder(pergunta_usuario, historico, contexto_extra="", ja_pesquisou=False):
    if not ja_pesquisou:
        prompt_sistema = (
            "Você é o assistente de busca.\n"
            "REGRA 1: Você NÃO SABE informações de eventos recentes, cotações ou notícias.\n"
            "REGRA 2: VOCÊ TEM SIM ACESSO À INTERNET. Tudo que requisitar pesquisas, pesquise.\n"
            "REGRA 3: Quando não souber, use EXATAMENTE a tag [PESQUISAR] e NADA MAIS na resposta final.\n\n"
            "EXEMPLOS OBRIGATÓRIOS DE COMO VOCÊ DEVE AGIR:\n"
            "Usuário: Qual a cotação do dólar hoje?\n"
            "Assistente: <think> Eu não sei a cotação de hoje, preciso acessar a internet. </think>\n[PESQUISAR] cotação dólar hoje\n\n"
            "Usuário: Notícias da SpaceX.\n"
            "Assistente: <think> Isso é um evento recente, não tenho na memória. </think>\n[PESQUISAR] SpaceX últimas notícias\n\n"
            "Usuário: Quanto é 2+2?\n"
            "Assistente: <think> Isso é matemática básica, eu sei a resposta. </think>\nA resposta é 4."
        )
    else:
        # PÓS-PESQUISA: Forçamos ela a sair do <think>
        prompt_sistema = (
            "Você é um assistente lógico.\n"
            "REGRA 1: Use a tag <think> APENAS para analisar os dados em silêncio.\n"
            "REGRA 2: Você JÁ RECEBEU os dados da internet. É PROIBIDO usar [PESQUISAR] novamente.\n"
            "REGRA 3: OBRIGATÓRIO: Você DEVE fechar a tag com </think> e escrever a resposta final ao usuário logo abaixo dela."
        )

    # if not ja_pesquisou:
    #     prompt_sistema = (
    #         "Você é o assistente pessoal de engenharia.\n"
    #         "REGRA 1: Você NÃO SABE informações de eventos recentes, cotações ou notícias.\n"
    #         "REGRA 2: VOCÊ TEM SIM ACESSO À INTERNET. Tudo que requisitar pesquisas, pesquise.\n"
    #         "REGRA 3: Quando não souber de algo atual, use EXATAMENTE a tag [PESQUISAR] e NADA MAIS na resposta final.\n\n"
    #         "EXEMPLOS OBRIGATÓRIOS:\n"
    #         "Usuário: Qual a cotação do dólar hoje?\n"
    #         "Assistente: <think> Eu não sei a cotação de hoje, preciso acessar a internet. </think>\n[PESQUISAR] cotação dólar hoje"
    #     )
    # else:
    #     prompt_sistema = (
    #         "Você é um assistente lógico.\n"
    #         "REGRA 1: Use a tag <think> APENAS para analisar os dados em silêncio.\n"
    #         "REGRA 2: Você JÁ RECEBEU os dados da internet. É PROIBIDO usar [PESQUISAR] novamente.\n"
    #         "REGRA 3: OBRIGATÓRIO: Você DEVE fechar a tag com </think> e escrever a resposta final."
    #     )
        prompt_sistema += f"\n\n[TEXTO DA INTERNET]:\n{contexto_extra}"

    # 1. MONTANDO A MENTE DA IA (Sistema + Passado + Presente)
    mensagens = [{"role": "system", "content": prompt_sistema}]

    # Injeta a memória da conversa no prompt
    mensagens.extend(historico)

    # Adiciona a pergunta atual
    mensagens.append({"role": "user", "content": pergunta_usuario})

    prompt_formatado = tokenizer.apply_chat_template(mensagens, tokenize=False, add_generation_prompt=True)
    prompt_completo = prompt_formatado + "<think>\n"

    inputs = tokenizer(prompt_completo, return_tensors="pt").to(modelo_ia.device)

    with torch.no_grad():
        outputs = modelo_ia.generate(
            **inputs,
            max_new_tokens=400,
            do_sample=True,
            temperature=0.3,
            repetition_penalty=1.0,
            pad_token_id=tokenizer.eos_token_id
        )

    texto_bruto = tokenizer.decode(outputs[0], skip_special_tokens=True)
    pensamento = "Falha no raciocínio."
    resposta_final = texto_bruto
    texto_gerado = texto_bruto.split("<think>")[-1]

    if "</think>" in texto_gerado:
        partes = texto_gerado.split("</think>")
        pensamento = partes[0].strip()
        resposta_final = partes[1].replace("<resposta>", "").replace("</resposta>", "").strip()

        if resposta_final == "":
             resposta_final = "A IA respondeu apenas no pensamento: " + pensamento

   # ==========================================
    # 2. O INTERCEPTADOR DE PESQUISA (CORRIGIDO)
    # ==========================================
    # Mudamos de texto_bruto para texto_gerado para o Python não ler as próprias regras
    if "[PESQUISAR]" in texto_gerado and not ja_pesquisou:
        termo_busca = texto_gerado.split("[PESQUISAR]")[1].split("</think>")[0].strip()
        termo_busca = termo_busca.split("\n")[0].replace(']', '')

        dados_da_web = pesquisar_na_internet(termo_busca)
        print("   [🧠 Dados capturados! Reinjetando na mente da IA...]")

        # Chama de novo, repassando o mesmo histórico
        return pensar_e_responder(pergunta_usuario, historico, contexto_extra=dados_da_web, ja_pesquisou=True)

    return pensamento, resposta_final

# ==========================================
# O LOOP PRINCIPAL (Onde a memória mora)
# ==========================================
print("\n==================================================")
print("🧠 IA COM RACIOCÍNIO, INTERNET E MEMÓRIA INICIADA")
print("==================================================")

# A variável que guarda as lembranças
memoria_da_conversa = []

while True:
    texto = input("\nVocê: ")
    if texto.lower() == 'sair':
        break

    # Passamos a memória para a IA ler
    pensamento_ia, resposta_ia = pensar_e_responder(texto, memoria_da_conversa)

    print("\n\033[90m[LÓGICA INTERNA DA IA]:")
    print(f"{pensamento_ia}\033[0m")
    print(f"\n🤖 Resposta Final:\n{resposta_ia}")
    print("-" * 50)

    # SALVANDO A MEMÓRIA: Guardamos o que você disse e o que ela respondeu
    memoria_da_conversa.append({"role": "user", "content": texto})
    memoria_da_conversa.append({"role": "assistant", "content": resposta_ia})

    # A JANELA DESLIZANTE: Manter apenas as últimas 6 mensagens (3 suas e 3 dela)
    # Isso impede que o computador trave com excesso de texto
    if len(memoria_da_conversa) > 6:
        memoria_da_conversa = memoria_da_conversa[-6:]

🚀 Carregando a IA e preparando o Motor de Raciocínio...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]


🧠 IA COM RACIOCÍNIO, INTERNET E MEMÓRIA INICIADA

[LÓGICA INTERNA DA IA]:
Carlos, uma pessoa comum, e você, Carlos, com seu nome de usuário. Sobre sua fruta favorita, a maçã é uma fruta muito popular e adorada por muitas pessoas. Ela é rica em vitamina C e pode ser consumida em muitas formas, como em um sanduíche, em um smoothie ou mesmo cozida em um bolo. É uma fruta que pode ser encontrada em muitos lugares e é uma opção saudável e deliciosa. Como você gosta de comer maçãs?

🤖 Resposta Final:
A IA respondeu apenas no pensamento: Carlos, uma pessoa comum, e você, Carlos, com seu nome de usuário. Sobre sua fruta favorita, a maçã é uma fruta muito popular e adorada por muitas pessoas. Ela é rica em vitamina C e pode ser consumida em muitas formas, como em um sanduíche, em um smoothie ou mesmo cozida em um bolo. É uma fruta que pode ser encontrada em muitos lugares e é uma opção saudável e deliciosa. Como você gosta de comer maçãs?
--------------------------------------------------

[LÓ